### Title: 02b_generate_tree_taxonomy
### Purpose: Generate tree taxonomies for the mixed meals and the ingredients datasets. The taxaonomy is used to build the food trees for these datasets. Ingredient codes are needed to construct the OTU table (referred to as IFC table in DietDiveR). This version takes the FDA ingredient descriptions to work with the polyphenol data (mapped with FDA-FDD descriptions rather than the FNDDS ingredient descriptions)
### Author: Jules Larke
### Date: November 15, 2025

### Load packages

In [ ]:
import pandas as pd
import string

## Mixed Meals Taxonomy

### Load data

In [ ]:
# NodeLabelsMCT.txt comes from DietDiveR
food_id = pd.read_csv('../../data/02/NodeLabelsMCT.txt', sep='\t')

# food_tree.txt comes from 00_generate_datasets and contains the unique foodcodes in our dataset
food_codes = pd.read_csv('../../data/00/food_tree/food_tree.txt', sep='\t')

### Get the unique foodcodes and descriptions in our dataset

In [ ]:
# filter food codes for what is in the data subset. load data:
subset_codes = pd.read_csv('../../data/01/insulin_resistance/wweia_mixed_meals_long.tsv', sep='\t', usecols=['foodcode'])

# get unique codes
subset_codes = subset_codes.drop_duplicates()

# filter data
food_codes = food_codes[food_codes['FoodCode'].isin(subset_codes['foodcode'])]
food_codes = food_codes.copy()

### Format text for use with DietDiveR trees

In [ ]:
# get punctuation for text cleaning
punct = string.punctuation
punct = punct.replace('_', '')

# apply function to clean text. removes punctuation and replaces whitespaces with underscores: .str.replace(' ', '_')
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

food_codes['Main.food.description'] = food_codes['Main.food.description'].apply(lambda x: clean_text(x))
food_codes['Main.food.description'] = food_codes['Main.food.description'].str.replace(' ', '_')

# save ingredient tree taxonomy
food_codes.to_csv('../../data/02/wweia_foodcode_taxa_clean.txt', sep='\t', index=None)

## Ingredients Taxonomy
### This code is used to generate the Food Tree taxonomies for the FNDDS ingredient descriptions with the Insulin Resistance dataset

### Load data

In [ ]:
ingred_id = pd.read_csv('../../data/00/food_tree/ingredient_tree.txt', sep='\t')
ir_codes = pd.read_csv('../../data/01/insulin_resistance/wweia_ingredients_long.tsv', sep='\t', usecols=['ingred_desc','ingred_code']) # ir data

### Reformat and merge data: generate a file with the ingredients descriptions, their FoodID corresponding to taxonomy and ingredient code for matching back to the main dataset

In [ ]:
# rename column for merging
ingred_id = ingred_id.rename(columns={'Ingredient.category.description': 'Main.food.description'})

ir_codes = ir_codes.rename(columns={'ingred_desc': 'Main.food.description'})
ir_codes = ir_codes.rename(columns={'ingred_code': 'Ingredient code'})

# get unique ingredient descriptions
ir_codes = ir_codes.drop_duplicates(subset='Main.food.description')

# merge node labels and ingredient codes
ir_node_and_code = ingred_id.merge(ir_codes, on='Main.food.description', how='left')

# select features for tree taxonomy
ir_node_labels = ir_node_and_code[['Level.code', 'Main.food.description']]

# create copies for text cleaning
ir_node_labels = ir_node_labels.copy()
ir_node_and_code = ir_node_and_code.copy()

### Clean text and save

In [ ]:
# get punctuation for text cleaning
punct = string.punctuation
punct = punct.replace('_', '')

# apply function to clean text. removes punctuation and replaces whitespaces with underscores: .str.replace(' ', '_')
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

ir_node_labels['Main.food.description'] = ir_node_labels['Main.food.description'].apply(lambda x: clean_text(x))
ir_node_labels['Main.food.description'] = ir_node_labels['Main.food.description'].str.replace(' ', '_')

# save ingredient tree taxonomy
ir_node_labels.to_csv('../../data/02/node_labels_clean.txt', sep='\t', index=None)

### Now we output the full taxaonomy file to build the tree. This contains the higher taxonomy levels in addition to the leaf nodes.

In [ ]:
# drop NAs which correspond to levels that are not ingredients (leaves) and have no ingredient codes
ir_node_and_code = ir_node_and_code.dropna()
ir_node_and_code['Ingredient code'] = ir_node_and_code['Ingredient code'].astype(int) 

# rename
ir_node_and_code = ir_node_and_code.rename(columns={'Level.code':'FoodID'})

# as above, apply function to clean text
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

ir_node_and_code['Main.food.description'] = ir_node_and_code['Main.food.description'].apply(lambda x: clean_text(x))
ir_node_and_code['Main.food.description'] = ir_node_and_code['Main.food.description'].str.replace(' ', '_')

# save ingredient taxonomy linked codes for OTU tables
ir_node_and_code.to_csv('../../data/02/ir_ingredient_taxa_clean.txt', sep='\t', index=None)

### Now do the same for polyphenol dataset

### Load data

In [ ]:
# polyphenol dataset unique descriptions
poly_codes = pd.read_csv('../../data/01/polyphenol/polyphenol_ingred_desc.csv')
poly_codes = poly_codes.copy()

### Reformat and merge data: generate a file with the ingredients descriptions, their FoodID corresponding to taxonomy and ingredient code for matching back to the main dataset

In [ ]:
poly_codes = poly_codes.rename(columns={'ingred_desc':'Main.food.description'})
poly_node_and_code = poly_codes.merge(ingred_id, on='Main.food.description')
poly_node_and_code.rename(columns={'Level.code':'FoodID'}, inplace=True)

### Clean text and save

In [ ]:
# get punctuation for text cleaning
punct = string.punctuation
punct = punct.replace('_', '')

# apply function to clean text. removes punctuation and replaces whitespaces with underscores: .str.replace(' ', '_')
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

poly_node_and_code['Main.food.description'] = poly_node_and_code['Main.food.description'].apply(lambda x: clean_text(x))
poly_node_and_code['Main.food.description'] = poly_node_and_code['Main.food.description'].str.replace(' ', '_')

# rename columns
poly_node_and_code.rename(columns={'Level.code': 'FoodID', 'ingred_code': 'Ingredient code'}, inplace=True)

# sort values
poly_node_and_code['FoodID'] = poly_node_and_code['FoodID'].astype(str)
poly_node_and_code = poly_node_and_code.sort_values('FoodID')

# save ingredient tree taxonomy
poly_node_and_code[['FoodID', 'Main.food.description', 'Ingredient code']].to_csv('../../data/02/poly_ingredient_taxa_clean.txt', sep='\t', index=None)